# T1-01 EDA: 대상 통계표 프로파일

`notebooks/kosis_fetch_tables.ipynb`로 받은 표(`data/raw/kosis/`)를 B0 로더·전처리(`kosis_chat`)로 읽고,
공유 메타데이터(`data/통계표_메타데이터.jsonl`)의 분류 계층과 합쳐 표마다 같은 항목을 측정한다.

- 측정 항목은 `plans/T1-01.md`의 측정표(입력 형식, 크기, 분류 체계, 시점, 단위·측정 종류, 값·결측, 빠진 조합, 합계 정합성, 질문 가능성)를 따른다.
- 결과는 `data/eda/profile.json`(커밋 안 함)과 화면 표로 나온다. `docs/eda.md`에는 구조·개수·비율만 옮기고 셀 값은 쓰지 않는다.
- 자동 판정(측정 종류, 계 이름 추정 등)은 휴리스틱이라 사람이 확인한다.

In [ ]:
# Setup: repo root on sys.path, target tables, paths.
import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

root = Path.cwd()
while not (root / "kosis_chat").is_dir() and root != root.parent:
    root = root.parent
assert (root / "kosis_chat").is_dir(), "open this notebook from inside the repo"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from kosis_chat.data.loader import load_table
from kosis_chat.data.preprocessing import clean
from kosis_chat.data.schema import SchemaError
from kosis_chat.data.serializer import to_markdown

TABLE_IDS = ["DT_1B8000G", "DT_1B26006_A01", "DT_1B83A35", "DT_1BPA001", "DT_1B81A24"]
RAW_DIR = root / "data" / "raw" / "kosis"
META_PATH = root / "data" / "통계표_메타데이터.jsonl"
OUT_DIR = root / "data" / "eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_NAMES = re.compile(r"^(계|합계|총계|전체|전국|총인구|소계)$|^(총|전체)\s")  # name-based guess, compared with meta hierarchy
RATIO_UNITS = re.compile(r"%|퍼센트|율|비\b|당|‰")


def md_table(df):
    """Markdown table without the optional tabulate dependency."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

In [ ]:
# Shared metadata for the target tables: classification hierarchy (lvl, upItmId, leaf) per category code.
meta_by_id = {}
with META_PATH.open(encoding="utf-8-sig") as f:
    for line in f:
        rec = json.loads(line)
        if rec.get("table_id") in TABLE_IDS:
            meta_by_id[rec["table_id"]] = rec
print("meta found:", sorted(meta_by_id), "missing:", sorted(set(TABLE_IDS) - set(meta_by_id)))


def hierarchy(meta):
    """{class name: {code: {'name', 'lvl', 'up', 'leaf'}}} from the shared metadata."""
    out = {}
    for cls in meta.get("classifications") or []:
        name = cls.get("classNm") or cls.get("classId")
        codes = {}
        for it in cls.get("itmList") or []:
            codes[str(it.get("itmId"))] = {
                "name": it.get("scrKor"),
                "lvl": int(it["lvl"]) if str(it.get("lvl") or "").isdigit() else None,
                "up": str(it.get("upItmId")) if it.get("upItmId") not in (None, "") else None,
                "leaf": str(it.get("leaf")).lower() in ("true", "y", "1") if it.get("leaf") is not None else None,
            }
        out[name] = codes
    return out

In [ ]:
# Load every downloaded file of the target tables through the B0 loader and cleaner.
def table_files(tbl_id):
    return sorted(p for p in RAW_DIR.rglob(f"*_{tbl_id}_*.json") if not p.name.endswith(".meta.json"))


def to_frame(table):
    rows = []
    for c in table.cells:
        row = {"item": c.item, "period": c.period_raw, "value_raw": c.value_raw, "value": c.value,
               "flag": c.value_flag, "unit": c.unit}
        for k, v in c.dims.items():
            row[f"dim:{k}"] = v
        for k, v in c.dim_codes.items():
            row[f"code:{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


loaded = {}  # (tbl_id, file name) -> (TableInput, DataFrame) or problems
for tbl_id in TABLE_IDS:
    files = table_files(tbl_id)
    if not files:
        print(f"{tbl_id}: no downloaded file in {RAW_DIR.relative_to(root)} (run kosis_fetch_tables.ipynb first)")
    for path in files:
        try:
            t = clean(load_table(path))
            loaded[(tbl_id, path.name)] = (t, to_frame(t))
            print(f"OK   {path.name}: {len(t.cells):,} cells")
        except SchemaError as exc:
            loaded[(tbl_id, path.name)] = exc.problems
            print(f"FAIL {path.name}: {[str(p) for p in exc.problems[:3]]}")

In [ ]:
# Profile one loaded file. Every number here is computed from the data, never typed in.
def period_shape(p):
    return re.sub(r"\d", "9", p)


def decimals(raw):
    m = re.search(r"\.(\d+)", raw.replace(",", ""))
    return len(m.group(1)) if m else 0


def sum_check(df, dims, hier):
    """Parent value vs sum of its child codes, other dims/item/period fixed. Count-like units only."""
    checked = exact = near = 0
    worst = 0.0
    ok = df[df["flag"].isin(["ok", "provisional"])]
    for dim in dims:
        codes = hier.get(dim) or {}
        children = defaultdict(list)
        for code, info in codes.items():
            if info["up"] and info["up"] in codes:
                children[info["up"]].append(code)
        if not children or f"code:{dim}" not in ok:
            continue
        keys = [c for c in ok.columns if c.startswith("code:") and c != f"code:{dim}"] + ["item", "period"]
        for _, grp in ok.groupby(keys, dropna=False) if keys else [(None, ok)]:
            vals = dict(zip(grp[f"code:{dim}"], grp["value"]))
            for parent, kids in children.items():
                if parent not in vals or not all(k in vals for k in kids):
                    continue
                diff = abs(vals[parent] - sum(vals[k] for k in kids))
                dec = max(decimals(r) for r in grp["value_raw"])
                checked += 1
                exact += diff == 0
                near += diff <= 0.5 * len(kids) * 10 ** -dec  # each published child may be rounded by half a unit
                worst = max(worst, diff)
    return {"checked": checked, "exact": exact, "within_rounding": near, "max_abs_diff": worst}


def profile(tbl_id, fname, table, df):
    meta = meta_by_id.get(tbl_id, {})
    hier = hierarchy(meta)
    dims = table.dim_names
    units = Counter(df["unit"])
    units_per_item = df.groupby("item")["unit"].nunique()
    flags = Counter(df["flag"])
    unparsed = sorted(set(df.loc[df["flag"] == "unparsed", "value_raw"]))[:10]
    periods = sorted(set(df["period"]))
    dim_codes = {d: sorted(set(df[f"code:{d}"])) if f"code:{d}" in df else [] for d in dims}
    expected = len(set(df["item"])) * len(periods)
    for d in dims:
        expected *= max(1, len(dim_codes[d]))
    hier_rows = {}
    for d in dims:
        codes = hier.get(d) or {}
        seen = set(dim_codes[d])
        lvls = Counter(codes[c]["lvl"] for c in seen if c in codes)
        name_total = sorted({codes[c]["name"] for c in seen if c in codes and TOTAL_NAMES.search(codes[c]["name"] or "")})
        roots = sorted({codes[c]["name"] for c in seen if c in codes and not codes[c]["up"]})
        hier_rows[d] = {
            "codes_in_data": len(seen), "codes_in_meta": len(codes), "not_in_meta": len(seen - set(codes)),
            "levels": dict(sorted((k, v) for k, v in lvls.items() if k is not None)),
            "non_leaf_in_data": sum(1 for c in seen if c in codes and codes[c]["leaf"] is False),
            "roots": roots, "total_like_names": name_total,
        }
    ratio_like = any(RATIO_UNITS.search(u or "") for u in units)
    md = to_markdown(table)
    leafy = [d for d in dims if hier_rows[d]["codes_in_data"] - hier_rows[d]["non_leaf_in_data"] >= 2]
    return {
        "tbl_id": tbl_id, "file": fname, "tbl_nm": table.meta.tbl_nm, "survey": table.meta.survey_nm,
        "period_type": table.meta.period_type,
        "size": {"cells": len(df), "items": df["item"].nunique(), "dims": dims,
                 "codes_per_dim": {d: len(v) for d, v in dim_codes.items()},
                 "markdown_chars": len(md), "approx_tokens": len(md) // 2},
        "period": {"count": len(periods), "first": periods[0] if periods else None, "last": periods[-1] if periods else None,
                   "shapes": sorted({period_shape(p) for p in periods})},
        "unit": {"distinct": dict(units), "items_with_mixed_units": int((units_per_item > 1).sum()),
                 "blank_unit_cells": int(units.get("", 0)), "meta_unit": meta.get("unit"),
                 "measure_kind_guess": "ratio" if ratio_like else "count-like (check)"},
        "values": {"flags": dict(flags), "parsed_share": round(df["value"].notna().mean(), 4) if len(df) else None,
                   "unparsed_examples": unparsed,
                   "max_decimals_by_item": {k: int(v) for k, v in df.groupby("item")["value_raw"].agg(lambda s: max(map(decimals, s))).items()}},
        "missing_combos": {"expected": expected, "present": len(df), "missing_share": round(1 - len(df) / expected, 4) if expected else None},
        "hierarchy": hier_rows,
        "sum_check": None if ratio_like else sum_check(df, dims, hier),
        "questions": {"단순 수치": flags.get("ok", 0) > 0, "추이": len(periods) >= 2, "비교": bool(leafy),
                      "구성비": (not ratio_like) and any(hier_rows[d]["non_leaf_in_data"] > 0 for d in dims),
                      "최대최소": any(hier_rows[d]["codes_in_data"] - hier_rows[d]["non_leaf_in_data"] >= 3 for d in dims)},
    }


profiles = [profile(tid, fname, *v) for (tid, fname), v in loaded.items() if not isinstance(v, list)]
failures = {f"{tid}/{fname}": [str(p) for p in v] for (tid, fname), v in loaded.items() if isinstance(v, list)}
(OUT_DIR / "profile.json").write_text(json.dumps({"profiles": profiles, "load_failures": failures},
                                                 ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"{len(profiles)} profiles -> {(OUT_DIR / 'profile.json').relative_to(root)}; load failures: {len(failures)}")

## 표 간 비교 (docs/eda.md에 옮길 표)

In [ ]:
# One row per file: the comparison table for docs/eda.md (structure and counts only).
summary = pd.DataFrame([{
    "표": p["tbl_id"], "주기": p["period_type"], "셀": p["size"]["cells"], "항목": p["size"]["items"],
    "분류": " × ".join(f"{d}({n})" for d, n in p["size"]["codes_per_dim"].items()),
    "시점": f"{p['period']['first']}~{p['period']['last']} ({p['period']['count']})",
    "단위": ", ".join(u or "(빈칸)" for u in p["unit"]["distinct"]),
    "결측 기호": ", ".join(f"{k} {v}" for k, v in p["values"]["flags"].items() if k != "ok") or "없음",
    "빠진 조합": f"{p['missing_combos']['missing_share']:.1%}" if p["missing_combos"]["missing_share"] is not None else "-",
    "합계 일치(정확/반올림 내/검사)": (f"{p['sum_check']['exact']}/{p['sum_check']['within_rounding']}/{p['sum_check']['checked']}"
                                   if p["sum_check"] else "비율 표(해당 없음)"),
    "토큰(대략)": p["size"]["approx_tokens"],
} for p in profiles])
print(md_table(summary) if len(summary) else "no profiles")

In [ ]:
# Question-type feasibility matrix (input for T1-02) and hierarchy details (input for R2 / T1-05).
print(md_table(pd.DataFrame([{"표": p["tbl_id"], **p["questions"]} for p in profiles])))
for p in profiles:
    print(f"\n## {p['tbl_id']} {p['tbl_nm']} ({p['file']})")
    for d, h in p["hierarchy"].items():
        print(f"- {d}: data {h['codes_in_data']} / meta {h['codes_in_meta']} codes, not in meta {h['not_in_meta']}, "
              f"levels {h['levels']}, non-leaf {h['non_leaf_in_data']}, roots {h['roots']}, total-like names {h['total_like_names']}")
    print(f"- unit: {p['unit']}")
    print(f"- unparsed examples: {p['values']['unparsed_examples']}")